# 🦷 DENTEX 4-Class Abnormal Tooth Detection Benchmark (Kaggle 1-Click)

**GitHub Repository:** [https://github.com/tmanhococ/dentex-benchmark](https://github.com/tmanhococ/dentex-benchmark)  
**Authors:** Dental Pathology AI Research Group  
**Target Runtime:** Kaggle Notebooks (GPU P100 / T4)

---

## 📖 Phần I: Tổng Quan Bài Toán & Phân Loại Bệnh Lý Lâm Sàng

Bài toán tập trung vào phát hiện và phân loại **4 tổn thương / bất thường răng miệng phổ biến nhất** trên phim chụp X-quang răng toàn cảnh (panoramic dental radiograph) từ bộ dữ liệu **DENTEX Challenge** (MICCAI 2023):

1. **Caries (ID: 1):** Sâu răng giai đoạn sớm / tổn thương men răng và ngà răng nông.
2. **Deep Caries (ID: 2):** Sâu răng mức độ nặng, tổn thương sâu tiến sát hoặc đã thâm nhiễm vào buồng tủy (pulp chamber).
3. **Periapical Lesion (ID: 3):** Tổn thương thấu quang quanh chóp / u nang viêm chân răng (radiolucency / cyst tại root apex).
4. **Impacted Tooth (ID: 4):** Răng ngầm / răng khôn kẹt cơ học không thể mọc bình thường lên cung hàm.

*Lưu ý y khoa:* Răng khỏe mạnh bình thường đóng vai trò là hậu cảnh (background) để mô hình tập trung phân biệt và định vị chính xác bệnh lý lâm sàng.


## 🔬 Phần II: Cơ Sở Lý Thuyết & Công Thức Toán Học Của Mô Hình

Notebook này triển khai và đối sánh 2 kiến trúc tiên tiến đại diện cho 2 trường phái giải thuật:

### 1. HierarchicalNet (Multi-Scale Feature Hierarchy)
Do các tổn thương nha khoa có sự chênh lệch kích thước rất lớn ($Caries$ có thể chỉ chiếm vài chục pixel trong khi $Impacted\ Tooth$ hay nang chân răng lớn chiếm hàng trăm pixel), **HierarchicalNet** sử dụng Feature Pyramid Network (FPN) trên nền xương sống ResNet-50.

#### a. Trích xuất đặc trưng Bottom-Up (ResNet-50 Backbone)
Ảnh đầu vào $I \in \mathbb{R}^{3 \times H \times W}$ đi qua 4 tầng dư với độ thu nhỏ lũy tiến:
$$C_2, C_3, C_4, C_5 = \text{ResNet50}(I)$$
trong đó số kênh tương ứng là $256, 512, 1024, 2048$ và stride lần lượt là $4, 8, 16, 32$.

#### b. Kết nối ngang & Đường truyền Top-Down (FPN)
Các đặc trưng ngữ nghĩa mức cao được truyền ngược xuống và hợp nhất với đặc trưng không gian chi tiết mức thấp:
$$P_5 = \text{Conv}_{3\times3}(\text{Conv}_{1\times1}(C_5))$$
$$P_i = \text{Conv}_{3\times3}\left(\text{Conv}_{1\times1}(C_i) + \text{Upsample}_{2\times}(P_{i+1})\right), \quad i \in \{4, 3, 2\}$$
Tất cả các tầng kim tự tháp $\{P_2, P_3, P_4, P_5\}$ đều có cùng số kênh chuẩn hóa $d = 256$.

#### c. Đầu dự đoán phân cấp & Hàm mất mát (Loss Function)
Đầu dự đoán phân cấp tổng hợp ngữ cảnh đa tỉ lệ và ánh xạ ra xác suất lớp $\hat{p} \in \mathbb{R}^{K+1}$ ($K=4$ bệnh lý + 1 background) cùng tọa độ box chuẩn hóa $\hat{b} \in [0, 1]^4$:
$$\mathcal{L}_{HierarchicalNet} = \mathcal{L}_{CE}(\hat{p}, p^*) + \lambda_{reg} \cdot \mathbf{1}_{[p^* \ge 1]} \mathcal{L}_{L1}(\hat{b}, b^*)$$

---

### 2. DiffusionNet (Progressive Denoising Diffusion Detector)
**DiffusionNet** tiếp cận bài toán phát hiện vật thể dưới dạng tiến trình khuếch tán xác suất ngược (Reverse Denoising Diffusion Process), biến đổi các bounding box ngẫu nhiên từ phân phối chuẩn thành các box chuẩn xác bao quanh bệnh lý.

#### a. Tiến trình khuếch tán thuận (Forward Noising Process)
Cho bounding box gốc $z_0 \in [0, 1]^4$. Tại bước thời gian liên tục $t \in [0, T]$ ($T=1000$):
$$q(z_t | z_0) = \mathcal{N}\left(z_t; \sqrt{\bar{\alpha}_t} z_0, (1 - \bar{\alpha}_t) \mathbf{I}\right)$$
với $\alpha_t = 1 - \beta_t$ và $\bar{\alpha}_t = \prod_{s=1}^t \alpha_s$.

#### b. Nhúng bước thời gian Sinusoidal & Điều kiện hóa
Bước thời gian $t$ được mã hóa qua hàm điều hòa:
$$e_k(t) = \begin{cases} \sin\left(t / 10000^{2k/d}\right), & k \text{ chẵn} \\ \cos\left(t / 10000^{2k/d}\right), & k \text{ lẻ} \end{cases}$$
Biểu diễn điều kiện hóa kết hợp đặc trưng toàn cục $F = \text{Backbone}(I)$ và vector thời gian:
$$Cond = \text{GlobalPool}(F) + \text{MLP}(e(t))$$

#### c. Khử nhiễu tiến trình nghịch đảo & Hàm mất mát
Mạng `DynamicDiffusionHead` ước lượng box sạch $\hat{z}_0$ và phân loại tổn thương:
$$\hat{z}_0, \hat{c} = f_\theta(z_t, Cond)$$
$$\mathcal{L}_{DiffusionNet} = \mathcal{L}_{cls}(\hat{c}, c^*) + \lambda_{L1} \|\hat{z}_0 - z_0\|_1 + \lambda_{giou} \mathcal{L}_{giou}(\hat{z}_0, z_0)$$


## ⚙️ Phần III: Quy Trình Huấn Luyện & Cơ Chế Chống Tràn VRAM / NaN

Để giải quyết triệt để lỗi **CUDA Out of Memory (OOM)** trên Kaggle P100/T4 (16 GB) và lỗi **Loss: NaN** khi fine-tune, hệ thống áp dụng các tiêu chuẩn ổn định công nghiệp kế thừa từ template `cv-dinoresnet.ipynb`:

```
[ Ảnh Panorama Gốc ~2800x1300 px ]
               │
               ▼
[ Multi-Scale Resize: Short Edge 480..800 px, Max Edge 1333 px ]
               │
               ▼
[ DataLoader: batch_size = 1, num_workers = 2 ]
               │
               ▼
[ AMP Autocast: torch.amp.autocast('cuda') (FP16/BF16) ]
               │
               ▼
[ Forward Pass: HierarchicalNet / DiffusionNet ]
               │
               ▼
[ Loss Calculation & Chia Cho Gradient Accumulation (16) ]
               │
               ▼
[ Scaled Backward: scaler.scale(loss).backward() ]
               │
               ▼
     [ Step % 16 == 0 ? ]
     ├── YES ──► [ scaler.unscale_(optimizer) ]
     │                │
     │                ▼
     │           [ Kiểm Tra Gradients Hữu Hạn ]
     │           ├── Inf/NaN ──► [ Bỏ qua batch lỗi, giảm scale, tránh NaN model ]
     │           └── Finite  ──► [ clip_grad_norm_(0.1) ──► optimizer.step() ]
     └── NO  ──► Tiếp tục tích lũy gradient
               │
               ▼
[ Cuối Mỗi Epoch (Max 30 Epochs) ]
               │
               ├── val_loss tốt nhất ──► Lưu outputs/<model>/best.pth
               └── Luôn cập nhật      ──► Lưu outputs/<model>/last.pth
```

- **Bộ nhớ tiêu thụ:** Nhờ `batch_size = 1` kết hợp tích lũy gradient 16 bước, VRAM thực tế luôn duy trì $< 6$ GB, chạy mượt mà trên GPU Kaggle P100/T4 mà không bao giờ gặp OOM.
- **Ổn định số học:** Gradient clipping $0.1$ và cơ chế unscale guard loại bỏ hoàn toàn hiện tượng bùng nổ gradient gây NaN.
- **Giới hạn số Epoch:** Huấn luyện tối đa **30 epochs**, tự động ghi nhận lại checkpoint tối ưu nhất (`best.pth`) và checkpoint cuối cùng (`last.pth`).


## 📊 Phần IV: Lý Thuyết Các Chỉ Số Đánh Giá (Evaluation Metrics)

### 1. Intersection over Union (IoU)
$$\text{IoU}(B_p, B_{gt}) = \frac{\text{Area}(B_p \cap B_{gt})}{\text{Area}(B_p \cup B_{gt})}$$

### 2. Average Precision (AP) & COCO Metrics
Đường cong Precision-Recall $P(r)$ cho từng lớp bệnh lý:
$$\text{AP}_c = \int_0^1 P_c(r) dr \approx \sum_{k=1}^N (R_k - R_{k-1}) \max_{\tilde{r} \ge R_k} P(\tilde{r})$$
- **$mAP_{50}$:** Trung bình AP của 4 bệnh lý tại ngưỡng $\text{IoU} = 0.50$.
- **$mAP_{75}$:** Ngưỡng đánh giá khắt khe tại $\text{IoU} = 0.75$.
- **$mAP_{50:95}$:** Chuẩn mực quốc tế COCO, tính trung bình qua 10 ngưỡng IoU từ $0.50$ đến $0.95$ (bước nhảy $0.05$):
  $$mAP_{50:95} = \frac{1}{10} \sum_{\tau \in \{0.50, 0.55, \dots, 0.95\}} mAP_\tau$$
- **Average Recall (AR@100):** Tỉ lệ phát hiện tổn thương tối đa khi cho phép mô hình dự đoán tối đa 100 boxes mỗi ảnh.

### 3. Chỉ Số Phân Loại Lâm Sàng (Clinical F1-Score)
Khớp dự đoán với Ground Truth ở ngưỡng $\text{IoU} \ge 0.50$ để xác định $TP, FP, FN$:
$$\text{Precision}_c = \frac{TP_c}{TP_c + FP_c}, \quad \text{Recall}_c = \frac{TP_c}{TP_c + FN_c}$$
$$\text{F1}_{macro} = \frac{1}{4} \sum_{c=1}^4 \frac{2 \cdot \text{Precision}_c \cdot \text{Recall}_c}{\text{Precision}_c + \text{Recall}_c}$$


## 📥 Cell 1: Clone Mã Nguồn từ GitHub vào Kaggle Working


In [ ]:
import os, sys, shutil
from pathlib import Path

KAGGLE_WORKING = Path("/kaggle/working/Dentex")
REPO_URL = "https://github.com/tmanhococ/dentex-benchmark.git"

if KAGGLE_WORKING.exists():
    shutil.rmtree(KAGGLE_WORKING)

print(f"🚀 Đang clone mã nguồn từ {REPO_URL}...")
!git clone {REPO_URL} {KAGGLE_WORKING}

%cd {KAGGLE_WORKING}
if str(KAGGLE_WORKING) not in sys.path:
    sys.path.insert(0, str(KAGGLE_WORKING))

print(f"✅ Thư mục làm việc hiện tại: {os.getcwd()}")


## 🛠️ Cell 2: Kiểm Tra Phần Cứng GPU & Cài Đặt Thư Viện


In [ ]:
!pip install --quiet pycocotools albumentations tabulate
!python scripts/setup_env.py


## 🗂️ Cell 3: Chuẩn Bị Dữ Liệu DENTEX (Smart Auto-Detection & COCO 4-Class)


In [ ]:
# Kiểm tra nếu đã gắn dataset DENTEX trên Kaggle Input (VD: rabiayilmaz/dentex-challenge-2023)
kaggle_input = Path("/kaggle/input")
dentex_inputs = list(kaggle_input.glob("*dentex*")) if kaggle_input.exists() else []

if dentex_inputs:
    data_src = dentex_inputs[0]
    print(f"📦 Phát hiện dataset có sẵn tại Kaggle Input: {data_src}")
    !python scripts/download_and_prepare.py --data_dir "{data_src}" --seed 42
else:
    print("🌐 Không tìm thấy Kaggle Input, tự động tải trực tiếp từ Hugging Face Hub (ibrahimhamamci/DENTEX)...")
    !python scripts/download_and_prepare.py --data_dir data --seed 42


## 🏋️ Cell 4: Huấn Luyện 2 Mô Hình Mục Tiêu (HierarchicalNet & DiffusionNet, 30 Epochs)


In [ ]:
print("\n==========================================================")
print("🏋️ [1/2] HUẤN LUYỆN HIERARCHICALNET (Multi-Scale Feature Hierarchy)...")
print("==========================================================")
!python scripts/train.py --model hierarchical_net --epochs 30 --lr 0.0001

print("\n==========================================================")
print("🏋️ [2/2] HUẤN LUYỆN DIFFUSIONNET (Progressive Denoising Diffusion)...")
print("==========================================================")
!python scripts/train.py --model diffusion_net --epochs 30 --lr 0.0001


## 📈 Cell 5: Đánh Giá Benchmark & Xuất Báo Cáo Khoa Học (Markdown & LaTeX)


In [ ]:
!python scripts/evaluate.py --compare_all

from IPython.display import display, Image as DisplayImage, Markdown

# Hiển thị đồ thị huấn luyện Loss
for m_key in ["hierarchical_net", "diffusion_net"]:
    curve_file = Path(f"outputs/{m_key}/training_curves.png")
    if curve_file.exists():
        print(f"📈 Đồ thị huấn luyện Loss: {m_key}")
        display(DisplayImage(str(curve_file)))

# Hiển thị biểu đồ so sánh
comp_plot = Path("outputs/metrics_comparison.png")
if comp_plot.exists():
    print("📊 Biểu đồ so sánh hiệu năng:")
    display(DisplayImage(str(comp_plot)))

summary_file = Path("outputs/benchmark_summary.md")
if summary_file.exists():
    print("📋 Bảng tổng hợp kết quả (Markdown):")
    display(Markdown(summary_file.read_text(encoding="utf-8")))

tex_file = Path("outputs/benchmark_table.tex")
if tex_file.exists():
    print("\n==========================================================")
    print("📄 MÃ NGUỒN BẢNG LATEX (Dùng cho bài báo khoa học):")
    print("==========================================================\n")
    print(tex_file.read_text(encoding="utf-8"))


## 🖼️ Cell 6: Trực Quan Hóa Bounding Box Dự Đoán Trên Ảnh X-Quang Thực Tế


In [ ]:
import cv2
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from dentex_bench.config import BenchmarkConfig
from dentex_bench.models import HierarchicalNetRunner, DiffusionNetRunner

CLASS_COLORS = {
    1: ('#E69F00', 'Caries'),
    2: ('#D55E00', 'Deep Caries'),
    3: ('#0072B2', 'Periapical Lesion'),
    4: ('#009E73', 'Impacted Tooth')
}

sample_images = list(Path("data").glob("**/*.png")) + list(Path("data").glob("**/*.jpg"))
if sample_images:
    test_img = sample_images[0]
    print(f"🖼️ Hiển thị dự đoán trên ảnh mẫu: {test_img}")

    cfg = BenchmarkConfig.from_auto_env()
    bgr = cv2.imread(str(test_img))
    if bgr is not None:
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        tensor = torch.from_numpy(rgb).permute(2, 0, 1).float() / 255.0

        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        runner_specs = [
            ("hierarchical_net", "HierarchicalNet", HierarchicalNetRunner),
            ("diffusion_net", "DiffusionNet", DiffusionNetRunner)
        ]

        for ax, (m_key, title, runner_cls) in zip(axes, runner_specs):
            ax.imshow(rgb)
            ax.set_title(f"{title} Predictions (Score >= 0.3)", fontsize=13)
            ax.axis("off")

            ckpt = Path(f"outputs/{m_key}/best.pth")
            if ckpt.exists():
                runner = runner_cls(cfg)
                runner.build_model()
                runner.load_checkpoint(ckpt)
                preds = runner.predict([tensor])[0]

                keep = preds["scores"] >= 0.3
                boxes = preds["boxes"][keep].cpu().numpy()
                labels = preds["labels"][keep].cpu().numpy()
                scores = preds["scores"][keep].cpu().numpy()

                for b, l, s in zip(boxes, labels, scores):
                    color, cname = CLASS_COLORS.get(int(l), ('#ffffff', f'Class {l}'))
                    x1, y1, x2, y2 = b
                    rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor=color, facecolor='none')
                    ax.add_patch(rect)
                    ax.text(x1, max(0, y1 - 5), f"{cname}: {s:.2f}", color='white',
                            bbox=dict(boxstyle='square,pad=0.2', facecolor=color, alpha=0.85), fontsize=9)

        plt.tight_layout()
        plt.show()
